In [ ]:
%pip install -U langsmith openevals requests python-dotenv

In [ ]:
import os

print(os.getcwd())

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print("LangSmith key:", bool(os.getenv("LANGSMITH_API_KEY")))
print("OpenAI key:", bool(os.getenv("OPENAI_API_KEY")))

In [ ]:
%pip install -U  openevals  

In [ ]:
import json
import base64
import requests
from pathlib import Path

from langsmith import Client
from openevals.llm import create_llm_as_judge


import sys

project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "agents" / "research_agent" / "research_analyzer.py").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not locate the project root.")

sys.path.insert(0, str(project_root))

from agents.research_agent.research_analyzer import _analyze_single_content

JSON_PATH = project_root / "results" / "dearduck_sa_research_20260922_112749.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    report = json.load(f)

print("Restaurant:", report["restaurant"]["name"])
print("Number of content items:", len(report["content"]))

In [ ]:
item = report["content"][0]

print("CONTENT ID:")
print(item["content_id"])

print("\nCAPTION:")
print(item["raw_data"]["caption"])

print("\nCURRENT ANALYSIS:")
print(
    json.dumps(
        item["analysis"],
        indent=2,
        ensure_ascii=False
    )
)

LangSmith Dataset

In [ ]:
client = Client()

DATASET_NAME = "rawaj-research-agent-judge-v1"

In [ ]:
examples = []

for item in report["content"]:

    content_input = {
        "content_id": item.get("content_id"),
        "short_code": item.get("short_code"),

        "raw_data": item.get("raw_data"),

        "reel_details": item.get("reel_details"),
    }

    examples.append(
        {
            "inputs": {
                "content": content_input
            },

            "metadata": {
                "restaurant": report["restaurant"]["name"],
                "content_id": item.get("content_id"),
                "short_code": item.get("short_code"),
            },
        }
    )

print("Examples prepared:", len(examples))

Testing 3 Posts

In [ ]:
test_examples = examples[:3]

len(test_examples)

Run this cell only once

In [ ]:
dataset = client.create_dataset(
    dataset_name=DATASET_NAME,
    description=(
        "Raw Instagram evidence used to evaluate "
        "the Rawaj Research Agent using LLM-as-a-judge."
    ),
)

client.create_examples(
    dataset_id=dataset.id,
    examples=test_examples,
)

print("Dataset created:", DATASET_NAME)

In [ ]:
JUDGE_PROMPT = """
You are evaluating Rawaj's Instagram Research Agent.

The INPUT contains the original Instagram evidence available
to the Research Agent.

The OUTPUT contains the analysis produced by the Research Agent.

You must determine whether the analysis is accurate, grounded
in the supplied evidence, and appropriate for a Research Agent.

Evaluate the following:

1. EVIDENCE GROUNDING
Every factual claim should be supported by the caption,
transcript, metadata, or supplied image evidence.

2. VISUAL ACCURACY
Check whether claims such as:
- food_visible
- menu_visible
- price_visible
- offer_visible
- logo_visible
- branding_visible
- people_visible
- text_in_visual
- visual_summary

are consistent with the supplied images.

3. CONTENT INTERPRETATION
Check whether:
- content_pillar
- content_categories
- content_themes
- CTA
- promotion

are reasonable interpretations of the evidence.

Do not require exact wording when two labels have essentially
the same meaning.

4. SUMMARY FAITHFULNESS
caption_summary, visual_summary and transcript_summary
must not invent information.

5. EVIDENCE QUALITY
Evidence items should genuinely support the fields they claim
to support.

6. RESEARCH ROLE COMPLIANCE
The Research Agent may observe, describe and analyze evidence.

It must NOT:
- qualify the restaurant
- decide whether it is a good or bad prospect
- assign a prospect score
- recommend marketing improvements
- identify marketing gaps
- create a marketing strategy

Return PASS when the analysis is materially accurate,
grounded and stays within the Research Agent role.

Return FAIL when there is a meaningful hallucination,
unsupported claim, incorrect interpretation, visual error,
or role violation.

Briefly explain your decision.

<input>
{inputs}
</input>

<output>
{outputs}
</output>

<images>
{attachments}
</images>
"""

In [ ]:
judge = create_llm_as_judge(
    prompt=JUDGE_PROMPT,
    model="gpt-5.6-terra",
    feedback_key="research_quality",
)

Image conversion helper

In [ ]:
import base64
import requests


def image_url_to_attachment(url: str):
    try:
        response = requests.get(
            url,
            timeout=30,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        response.raise_for_status()

        mime_type = (
            response.headers
            .get("Content-Type", "image/jpeg")
            .split(";")[0]
            .lower()
        )

        # Only formats supported by OpenEvals/OpenAI vision
        supported_types = {
            "image/jpeg",
            "image/png",
            "image/gif",
            "image/webp",
        }

        if mime_type not in supported_types:
            print(
                f"⚠️ Unsupported response type: {mime_type}"
            )
            return None

        encoded = base64.b64encode(
            response.content
        ).decode("utf-8")

        # IMPORTANT:
        # This must be a complete data URI.
        data_uri = (
            f"data:{mime_type};base64,{encoded}"
        )

        return {
            "mime_type": mime_type,
            "data": data_uri,
        }

    except Exception as e:
        print(f"⚠️ Could not load image: {e}")
        return None

In [ ]:
#image loading test

first_image_url = (
    report["content"][0]
    ["raw_data"]
    ["image_urls"][0]
)

attachment = image_url_to_attachment(
    first_image_url
)

print("Loaded:", attachment is not None)

if attachment:
    print("Mime type:", attachment["mime_type"])
    print(
        "Beginning:",
        attachment["data"][:50]
    )


In [ ]:
#LLM-as-a-judge evaluation function
def research_quality_evaluator(
    inputs: dict,
    outputs: dict,
    reference_outputs=None,
):

    content = inputs["content"]

    raw_data = content.get(
        "raw_data",
        {}
    )

    image_urls = raw_data.get(
        "image_urls",
        []
    )

    attachments = []

    for url in image_urls:

        attachment = image_url_to_attachment(
            url
        )

        if attachment is not None:
            attachments.append(
                attachment
            )

    result = judge(
        inputs=inputs,
        outputs=outputs,
        attachments=attachments,
    )

    return result

In [ ]:
#target function for evaluation
def target(inputs: dict) -> dict:

    content = inputs["content"]

    result = _analyze_single_content(
        content
    )

    if hasattr(result, "model_dump"):
        result = result.model_dump()

    return {
        "analysis": result
    }

In [ ]:
#test the target function with the first example
test_input = test_examples[0]["inputs"]

result = target(test_input)

print(
    json.dumps(
        result,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
judge_result = research_quality_evaluator(
    inputs=test_input,
    outputs=result,
)

judge_result

In [ ]:
results = client.evaluate(
    target,
    data=DATASET_NAME,

    evaluators=[
        research_quality_evaluator
    ],

    experiment_prefix="rawaj-research-agent-judge-test",

    description=(
        "Multimodal LLM-as-a-judge evaluation "
        "of Rawaj Research Agent."
    ),

    max_concurrency=0,
)